<a href="https://colab.research.google.com/github/IreneAbbey/The-Logistics-Auditor/blob/dev/Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd

# **Story 1: The Schema Builder**

In [5]:
#Load datasets
orders = pd.read_csv('olist_orders_dataset.csv')
reviews = pd.read_csv('olist_order_reviews_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')

In [6]:
master_ds = orders.merge(reviews, on='order_id', how='inner') \
                  .merge(customers, on='customer_id', how='inner')

In [8]:
#Ensure no duplicate rows were introduced by the joins
print("Duplicate rows:", master_ds.duplicated().sum())

Duplicate rows: 0


# **Story 2: The "Real" Delay Calculator**

In [43]:
#Convert columns to actual datetime objects
master_ds['order_estimated_delivery_date'] = pd.to_datetime(master_ds['order_estimated_delivery_date'])
master_ds['order_delivered_customer_date'] = pd.to_datetime(master_ds['order_delivered_customer_date'])

In [44]:
master_ds['Days_Difference'] = (
    master_ds['order_estimated_delivery_date'] - master_ds['order_delivered_customer_date']
).dt.days

In [71]:
def classify_with_flags(row):
    status = row['order_status']

    if status in ['canceled', 'unavailable']:
        return "Not Delivered"

    if pd.isna(row['order_delivered_customer_date']):
        return "In Progress / Unknown"

    days = row['Days_Difference']

    if days >= 0:
        return "On Time"
    elif days >= -5:
        return "Late"
    else:
        return "Super Late"

master_ds['Delivery_Status'] = master_ds.apply(classify_with_flags, axis=1)

In [72]:
print(master_ds)

                               order_id                       customer_id  \
0      e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1      53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2      47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3      949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4      ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   
...                                 ...                               ...   
99219  9c5dedf39a927c1b2549525ed64a053c  39bd1228ee8140590ac3aca26f2dfe00   
99220  63943bddc261676b46f01ca7ac2f7bd8  1fca14ff2861355f6e5f14306ff977a7   
99221  83c1379a015df1e13d02aae0204711ab  1aa71eb042121263aafbe80c1b562c9c   
99222  11c177c8e97725db2631073c19f07b62  b331b74b18dc79bcdf6532d51e1637c1   
99223  66dea50a8b16d9b4dee7af250b4be1a5  edb027a75a1449115f6b43211ae02a24   

      order_status order_purchase_timestamp    order_approved_at  \
0      